# Table of Contents
- [Description of notebook](#description)
- [Explore and format `essential_skills.csv`](#explore-and-format-essential_skillscsv)
    - [`essential_skills.csv` observations](#essential_skillscsv-observations)
- [Explore and format `job_zones.csv`](#explore-and-format-jobzonescsv)
    - [`job_zones.csv` observations](#jobzonescsv-observations)

# Description
This notebook is the workspace used to create the wide formatted CSV files for the `/data/onet_skills/essential_skills.csv` and `/data/onet_quals/jobzones.csv` files.

It conducts some exploration of the generate shape and format of the original O*NET CSV files first before doing processing to create the final `/data/onet_formatted/formatted_essential_skills.csv` and `/data/onet_formatted/formatted_jobzones.csv` files.

Note that the root (`/`) in the paths above refer to the project root.

# Explore and format `essential_skills.csv`

In [1]:
import pandas as pd

data_dir = "../../data/"
formatted_dir = data_dir + "onet_formatted/"
es_dir = data_dir + "onet_skills/"

# Read in CSV and get a feel of the shape
df_es = pd.read_csv(es_dir + "essential_skills.csv")
df_es.head(20)

,O*NET-SOC Code,Title,Element ID,Element Name,Scale ID,Scale Name,Data Value,N,Standard Error,Lower CI Bound,Upper CI Bound,Recommend Suppress,Not Relevant,Date,Domain Source
0,11-1011.00,Chief Executives,2.A.1.a,Reading Comprehension,IM,Importance,4.12,8,0.1250,3.8800,4.3700,N,NaN,08/2023,Analyst
1,11-1011.00,Chief Executives,2.A.1.a,Reading Comprehension,LV,Level,4.62,8,0.1830,4.2664,4.9836,N,N,08/2023,Analyst
2,11-1011.00,Chief Executives,2.A.1.b,Active Listening,IM,Importance,4.00,8,0.0000,4.0000,4.0000,N,NaN,08/2023,Analyst
3,11-1011.00,Chief Executives,2.A.1.b,Active Listening,LV,Level,4.75,8,0.1637,4.4292,5.0708,N,N,08/2023,Analyst
4,11-1011.00,Chief Executives,2.A.1.c,Writing,IM,Importance,4.12,8,0.1250,3.8800,4.3700,N,NaN,08/2023,Analyst
5,11-1011.00,Chief Executives,2.A.1.c,Writing,LV,Level,4.38,8,0.1830,4.0164,4.7336,N,N,08/2023,Analyst
6,11-1011.00,Chief Executives,2.A.1.d,Speaking,IM,Importance,4.25,8,0.1637,3.9292,4.5708,N,NaN,08/2023,Analyst
7,11-1011.00,Chief Executives,2.A.1.d,Speaking,LV,Level,4.75,8,0.1637,4.4292,5.0708,N,N,08/2023,Analyst
8,11-1011.00,Chief Executives,2.A.1.e,Mathematics,IM,Importance,3.25,8,0.2500,2.7600,3.7400,N,NaN,08/2023,Analyst
9,11-1011.00,Chief Executives,2.A.1.e,Mathematics,LV,Level,3.50,8,0.2673,2.9762,4.0238,N,N,08/2023,Analyst


In [2]:
# Get list of element names
es_elements = list(df_es["Element Name"].unique())
print(f"[INFO] Total types of elements: {len(es_elements)}")
es_elements

[INFO] Total types of elements: 10


['Reading Comprehension',
 'Active Listening',
 'Writing',
 'Speaking',
 'Mathematics',
 'Science',
 'Critical Thinking',
 'Active Learning',
 'Learning Strategies',
 'Monitoring']

In [3]:
# Get list of scales
es_scales = list(df_es["Scale Name"].unique())
print(f"[INFO] Total types of scales: {len(es_scales)}")
es_scales

[INFO] Total types of scales: 2


['Importance', 'Level']

## `essential_skills.csv` observations

### Shape
Looking at the resulting DataFrame, it looks like there are multiple element names. The shape seems to be the following: for each job title for each element for each scale, there is an associated data value. 

Therefore, formatting `essential_skills.csv` will require us to rank the elements using the associated data value. We will do this for each scale. 

In addition, we will have to pivot the vertically expanded score info (rows for each element score for each scale) into a horizontally expanded score info (cols for each element score for each scale).

### Expected size
It looks like there are 10 types of elements and 2 types of scales, so we can expect the final formatted DataFrame/CSV to have:
- $1$ column for each O*NET job code 
- $1$ column for each associated job title (if each code has exactly one job title)
- $2$ columns for the ranking of elements for each scale
- $2*10$ columns for data values only

For a total of $24$ columns.

In [4]:
# Get list of O*NET occupation codes and job titles
es_codes = list(df_es["O*NET-SOC Code"].unique())
es_titles = list(df_es["Title"].unique())
print(f"[INFO] Number of O*NET Codes: {len(es_codes)}, Number of job titles: {len(es_titles)}") 

# Validate that each code only has one job title
df_es_code_title = df_es[["O*NET-SOC Code", "Title"]].drop_duplicates()
mult_titles_for_es_code = df_es_code_title["O*NET-SOC Code"].duplicated().any()
if mult_titles_for_es_code:
    print("[WARNING] There are multiple titles associated with a job code")
else:
    print("[INFO] Each job code has one and only one associated title")

[INFO] Number of O*NET Codes: 910, Number of job titles: 910
[INFO] Each job code has one and only one associated title


In [5]:
df_es_code_title

,O*NET-SOC Code,Title
0,11-1011.00,Chief Executives
20,11-1011.03,Chief Sustainability Officers
40,11-1021.00,General and Operations Managers
60,11-2011.00,Advertising and Promotions Managers
80,11-2021.00,Marketing Managers
...,...,...
18100,53-7071.00,Gas Compressor and Gas Pumping Station Operators
18120,53-7072.00,"Pump Operators, Except Wellhead Pumpers"
18140,53-7073.00,Wellhead Pumpers
18160,53-7081.00,Refuse and Recyclable Material Collectors


In [6]:
# Aggregate the ranked data into one DataFrame
df_es_importance = (df_es[df_es["Scale Name"] == "Importance"]
            .sort_values(["O*NET-SOC Code", "Data Value"], ascending=[True, False])
            .groupby("O*NET-SOC Code")["Element Name"]
            .agg("|".join))

df_es_importance = df_es_importance.to_frame().reset_index()
df_es_importance.rename(
    columns={
        "Element Name" : "essential_skills_importance_ranked"
    }, 
    inplace=True)

In [7]:
# Normalize essential_skills_importance_ranked column vals
df_es_importance["essential_skills_importance_ranked"] = (
    df_es_importance["essential_skills_importance_ranked"]
        .str.lower()
        .str.replace(pat={
            " ": "_"
        })
)

df_es_importance.head()

,O*NET-SOC Code,essential_skills_importance_ranked
0,11-1011.00,critical_thinking|speaking|reading_comprehensi...
1,11-1011.03,writing|critical_thinking|reading_comprehensio...
2,11-1021.00,reading_comprehension|active_listening|speakin...
3,11-2011.00,active_listening|speaking|critical_thinking|re...
4,11-2021.00,active_listening|speaking|reading_comprehensio...


In [8]:
# Aggregate the ranked data into one DataFrame
df_es_level = (df_es[df_es["Scale Name"] == "Level"]
               .sort_values(["O*NET-SOC Code", "Data Value"], ascending=[True, False])
               .groupby("O*NET-SOC Code")["Element Name"]
               .agg("|".join))

df_es_level = df_es_level.to_frame().reset_index()
df_es_level.rename(columns={
        "Element Name": "essential_skills_level_ranked"
    }, 
    inplace=True)

df_es_level.head()

,O*NET-SOC Code,essential_skills_level_ranked
0,11-1011.00,Monitoring|Active Listening|Speaking|Critical ...
1,11-1011.03,Reading Comprehension|Writing|Speaking|Critica...
2,11-1021.00,Active Listening|Speaking|Monitoring|Reading C...
3,11-2011.00,Active Listening|Speaking|Critical Thinking|Ac...
4,11-2021.00,Reading Comprehension|Active Listening|Speakin...


In [9]:
# Normalize essential_skills_level_ranked column vals
df_es_level["essential_skills_level_ranked"] = (
    df_es_level["essential_skills_level_ranked"]
            .str.lower()
            .str.replace(pat={
                " ": "_"
            })
)

df_es_level

,O*NET-SOC Code,essential_skills_level_ranked
0,11-1011.00,monitoring|active_listening|speaking|critical_...
1,11-1011.03,reading_comprehension|writing|speaking|critica...
2,11-1021.00,active_listening|speaking|monitoring|reading_c...
3,11-2011.00,active_listening|speaking|critical_thinking|ac...
4,11-2021.00,reading_comprehension|active_listening|speakin...
...,...,...
905,53-7071.00,reading_comprehension|active_listening|speakin...
906,53-7072.00,monitoring|critical_thinking|reading_comprehen...
907,53-7073.00,monitoring|speaking|critical_thinking|reading_...
908,53-7081.00,critical_thinking|active_listening|reading_com...


In [10]:
# Merge DataFrames together
df_es_rankings = pd.merge(left=df_es_importance,
                          right=df_es_level,
                          on="O*NET-SOC Code")
df_es_rankings = pd.merge(left=df_es_code_title, 
                          right=df_es_rankings,
                          on="O*NET-SOC Code")

In [11]:
df_es_rankings.head()

,O*NET-SOC Code,Title,essential_skills_importance_ranked,essential_skills_level_ranked
0,11-1011.00,Chief Executives,critical_thinking|speaking|reading_comprehensi...,monitoring|active_listening|speaking|critical_...
1,11-1011.03,Chief Sustainability Officers,writing|critical_thinking|reading_comprehensio...,reading_comprehension|writing|speaking|critica...
2,11-1021.00,General and Operations Managers,reading_comprehension|active_listening|speakin...,active_listening|speaking|monitoring|reading_c...
3,11-2011.00,Advertising and Promotions Managers,active_listening|speaking|critical_thinking|re...,active_listening|speaking|critical_thinking|ac...
4,11-2021.00,Marketing Managers,active_listening|speaking|reading_comprehensio...,reading_comprehension|active_listening|speakin...


In [12]:
# Use pivot to create individual score columns for each element for each scale
df_es_scale_vals = df_es.pivot(index="O*NET-SOC Code",
                                   columns=["Element Name", "Scale Name"],
                                   values="Data Value")

# Rename columns
df_es_scale_vals.columns = [
    f"essential_skills_{elem.replace(" ", "_").lower()}_{scale.lower()}"
    for elem, scale in df_es_scale_vals.columns
]

# Reset the index
df_es_scale_vals = df_es_scale_vals.reset_index()

In [13]:
df_es_scale_vals.head()

,O*NET-SOC Code,essential_skills_reading_comprehension_importance,essential_skills_reading_comprehension_level,essential_skills_active_listening_importance,essential_skills_active_listening_level,essential_skills_writing_importance,essential_skills_writing_level,essential_skills_speaking_importance,essential_skills_speaking_level,essential_skills_mathematics_importance,...,essential_skills_science_importance,essential_skills_science_level,essential_skills_critical_thinking_importance,essential_skills_critical_thinking_level,essential_skills_active_learning_importance,essential_skills_active_learning_level,essential_skills_learning_strategies_importance,essential_skills_learning_strategies_level,essential_skills_monitoring_importance,essential_skills_monitoring_level
0,11-1011.00,4.12,4.62,4.00,4.75,4.12,4.38,4.25,4.75,3.25,...,1.62,0.75,4.38,4.75,3.75,4.50,3.12,3.75,4.00,5.25
1,11-1011.03,4.00,4.25,4.00,4.00,4.12,4.25,4.00,4.12,2.88,...,2.12,1.88,4.12,4.12,3.75,3.88,3.38,3.75,3.75,4.00
2,11-1021.00,4.00,4.00,4.00,4.12,3.50,3.88,4.00,4.12,2.62,...,1.50,0.62,3.88,3.88,3.62,3.75,3.00,3.12,4.00,4.12
3,11-2011.00,3.75,4.00,4.12,4.12,3.75,4.00,4.12,4.12,2.88,...,1.50,0.50,3.88,4.12,3.38,4.12,3.00,3.12,3.50,4.12
4,11-2021.00,3.88,4.12,4.12,4.12,3.25,3.88,4.00,4.12,3.00,...,1.75,1.25,3.88,4.12,3.88,4.12,3.25,3.50,3.88,4.12


In [14]:
# Merge this with the rankings DataFrame
df_es_formatted = pd.merge(right=df_es_scale_vals,
                           left=df_es_rankings,
                           on="O*NET-SOC Code")
df_es_formatted.rename(columns={
        "O*NET-SOC Code": "ONET_SOC_CODE",
        "Title": "title"
    }, 
    inplace=True)

In [15]:
df_es_formatted.head()

,ONET_SOC_CODE,title,essential_skills_importance_ranked,essential_skills_level_ranked,essential_skills_reading_comprehension_importance,essential_skills_reading_comprehension_level,essential_skills_active_listening_importance,essential_skills_active_listening_level,essential_skills_writing_importance,essential_skills_writing_level,...,essential_skills_science_importance,essential_skills_science_level,essential_skills_critical_thinking_importance,essential_skills_critical_thinking_level,essential_skills_active_learning_importance,essential_skills_active_learning_level,essential_skills_learning_strategies_importance,essential_skills_learning_strategies_level,essential_skills_monitoring_importance,essential_skills_monitoring_level
0,11-1011.00,Chief Executives,critical_thinking|speaking|reading_comprehensi...,monitoring|active_listening|speaking|critical_...,4.12,4.62,4.00,4.75,4.12,4.38,...,1.62,0.75,4.38,4.75,3.75,4.50,3.12,3.75,4.00,5.25
1,11-1011.03,Chief Sustainability Officers,writing|critical_thinking|reading_comprehensio...,reading_comprehension|writing|speaking|critica...,4.00,4.25,4.00,4.00,4.12,4.25,...,2.12,1.88,4.12,4.12,3.75,3.88,3.38,3.75,3.75,4.00
2,11-1021.00,General and Operations Managers,reading_comprehension|active_listening|speakin...,active_listening|speaking|monitoring|reading_c...,4.00,4.00,4.00,4.12,3.50,3.88,...,1.50,0.62,3.88,3.88,3.62,3.75,3.00,3.12,4.00,4.12
3,11-2011.00,Advertising and Promotions Managers,active_listening|speaking|critical_thinking|re...,active_listening|speaking|critical_thinking|ac...,3.75,4.00,4.12,4.12,3.75,4.00,...,1.50,0.50,3.88,4.12,3.38,4.12,3.00,3.12,3.50,4.12
4,11-2021.00,Marketing Managers,active_listening|speaking|reading_comprehensio...,reading_comprehension|active_listening|speakin...,3.88,4.12,4.12,4.12,3.25,3.88,...,1.75,1.25,3.88,4.12,3.88,4.12,3.25,3.50,3.88,4.12


In [16]:
# Write the formatted DataFrame to a final formatted CSV file
df_es_formatted.to_csv(formatted_dir + "formatted_essential_skills.csv", index=False)

# Explore and format `jobzones.csv`

In [17]:
jz_dir = data_dir + "onet_quals/"
df_jz = pd.read_csv(jz_dir + "job_zones.csv")

In [18]:
df_jz

,O*NET-SOC Code,Title,Job Zone,Date,Domain Source
0,11-1011.00,Chief Executives,5,08/2023,Analyst
1,11-1011.03,Chief Sustainability Officers,5,08/2021,Analyst
2,11-1021.00,General and Operations Managers,4,08/2023,Analyst
3,11-1031.00,Legislators,4,06/2008,Analyst
4,11-2011.00,Advertising and Promotions Managers,4,08/2026,Analyst
...,...,...,...,...,...
918,53-7071.00,Gas Compressor and Gas Pumping Station Operators,2,02/2026,Analyst
919,53-7072.00,"Pump Operators, Except Wellhead Pumpers",2,02/2026,Analyst
920,53-7073.00,Wellhead Pumpers,2,02/2026,Analyst
921,53-7081.00,Refuse and Recyclable Material Collectors,2,02/2026,Analyst


In [19]:
jz_codes = list(df_jz["O*NET-SOC Code"].unique())
jz_titles = list(df_jz["Title"].unique())

print(f"[INFO] Number of O*NET Codes: {len(jz_codes)}, Number of job titles: {len(jz_titles)}")

[INFO] Number of O*NET Codes: 923, Number of job titles: 923


In [20]:
# Validate that there is only one job title for each O*NET Code
df_jz_code_title = df_jz[["O*NET-SOC Code", "Title"]].drop_duplicates()
mult_titles_for_jz_code = df_jz_code_title["O*NET-SOC Code"].duplicated().any()
if mult_titles_for_jz_code:
    print("[WARNING] There are multiple titles associated with a job code")
else:
    print("[INFO] Each job code has one and only one associated title")

[INFO] Each job code has one and only one associated title


In [21]:
df_jz_code_title

,O*NET-SOC Code,Title
0,11-1011.00,Chief Executives
1,11-1011.03,Chief Sustainability Officers
2,11-1021.00,General and Operations Managers
3,11-1031.00,Legislators
4,11-2011.00,Advertising and Promotions Managers
...,...,...
918,53-7071.00,Gas Compressor and Gas Pumping Station Operators
919,53-7072.00,"Pump Operators, Except Wellhead Pumpers"
920,53-7073.00,Wellhead Pumpers
921,53-7081.00,Refuse and Recyclable Material Collectors


In [22]:
# See which job codes are in jobzones.csv but not essential_skills.csv and vice versa
set_jz_code_title = set(zip(df_jz_code_title["O*NET-SOC Code"], df_jz_code_title["Title"]))
set_es_code_title = set(zip(df_es_code_title["O*NET-SOC Code"], df_es_code_title["Title"]))

set_only_jz_code_title = set_jz_code_title - set_es_code_title
set_only_es_code_title = set_es_code_title - set_jz_code_title

In [23]:
set_only_jz_code_title

{('11-1031.00', 'Legislators'),
 ('13-2051.00', 'Financial and Investment Analysts'),
 ('13-2054.00', 'Financial Risk Specialists'),
 ('17-3028.00', 'Calibration Technologists and Technicians'),
 ('19-4044.00', 'Hydrologic Technicians'),
 ('27-4015.00', 'Lighting Technicians'),
 ('29-1212.00', 'Cardiologists'),
 ('29-1242.00', 'Orthopedic Surgeons, Except Pediatric'),
 ('29-1243.00', 'Pediatric Surgeons'),
 ('29-2042.00', 'Emergency Medical Technicians'),
 ('29-9021.00', 'Health Information Technologists and Medical Registrars'),
 ('39-4012.00', 'Crematory Operators'),
 ('53-3054.00', 'Taxi Drivers')}

In [24]:
set_only_es_code_title

set()

# `jobzones.csv` observations
## Shape
This looks like a simple formatting, as `jobzones.csv` looks like it only has one "item" of interest, which is the `Job Zone` column. This looks to be an ordinal data category, where as a `Job Zone` value increase corresponds to an increase in expected experience.

However, it looks like `jobzones.csv` contains occupations that are not contained in `essential_skills.csv`. 

The occupation codes and associated titles that are in `jobzones.csv` but not `essential_skills.csv` are the following:
```bash
{('11-1031.00', 'Legislators'),
 ('13-2051.00', 'Financial and Investment Analysts'),
 ('13-2054.00', 'Financial Risk Specialists'),
 ('17-3028.00', 'Calibration Technologists and Technicians'),
 ('19-4044.00', 'Hydrologic Technicians'),
 ('27-4015.00', 'Lighting Technicians'),
 ('29-1212.00', 'Cardiologists'),
 ('29-1242.00', 'Orthopedic Surgeons, Except Pediatric'),
 ('29-1243.00', 'Pediatric Surgeons'),
 ('29-2042.00', 'Emergency Medical Technicians'),
 ('29-9021.00', 'Health Information Technologists and Medical Registrars'),
 ('39-4012.00', 'Crematory Operators'),
 ('53-3054.00', 'Taxi Drivers')}
```

This observation might come in handy when we get to the exploratory data analysis (EDA) portion of the project.

In [25]:
df_jz.drop(columns=["Date", "Domain Source"], inplace=True)
df_jz.rename(columns={
        "O*NET-SOC Code" : "ONET_SOC_CODE",
        "Title": "title"
    }, 
    inplace=True)

In [26]:
df_jz

,ONET_SOC_CODE,title,Job Zone
0,11-1011.00,Chief Executives,5
1,11-1011.03,Chief Sustainability Officers,5
2,11-1021.00,General and Operations Managers,4
3,11-1031.00,Legislators,4
4,11-2011.00,Advertising and Promotions Managers,4
...,...,...,...
918,53-7071.00,Gas Compressor and Gas Pumping Station Operators,2
919,53-7072.00,"Pump Operators, Except Wellhead Pumpers",2
920,53-7073.00,Wellhead Pumpers,2
921,53-7081.00,Refuse and Recyclable Material Collectors,2


In [27]:
df_jz.to_csv(formatted_dir+"formatted_jobzones.csv", index=False)